# 05 – Bronze: incremental load from PostgreSQL into Iceberg tables

**Goal (Task 3, parts 2 and 3):**

1. Upgrade the Task 2 **incremental** load (`append` on `updated_at`) so it writes **Iceberg tables** registered in the catalog, in the **bronze** zone, **partitioned by ingestion date**.
2. Read the tables with **PyIceberg** and verify the data.
3. Add records in the app, rerun, and use **snapshots** to read the table **before and after** (time travel).
4. Bonus: the same incremental load into the additional pile (RustFS).

What changed compared to notebook 02:

| | Task 2 (notebook 02) | Task 3 (this notebook) |
|---|---|---|
| destination | `lake_destination("silo")` → bucket `pawhome-lake` | `pile_destination("silo")` → bucket `pawhome-pile` |
| format | Parquet files | **Iceberg tables** (`table_format="iceberg"`) |
| table | folder `postgres/animals_incremental/` | `bronze.postgres_animals` in the **catalog** |
| partition | in the key (`ingestion_date=` in the layout) | **Iceberg partition** `ingestion_date = day(_ingested_at)` |

In [ ]:
import dlt
import pandas as pd
import sqlalchemy as sa
from dlt.sources.sql_database import sql_table
from lake import POSTGRES_URL, DB_TABLES, PIPELINES_DIR, add_metadata
from pile import (BRONZE, get_catalog, pile_destination, partition_by_ingestion_date, run_to_pile,
                  ensure_pile_bucket, list_pile, show_catalog_db, snapshots, data_files)

pd.set_option("display.max_colwidth", 120)
engine = sa.create_engine(POSTGRES_URL)
catalog = get_catalog("silo")

## Step 1 – The resources

Same as notebook 02 (incremental on `updated_at` + metadata columns), plus:
- table name **`postgres_<table>`** → the source system is in the table name: `bronze.postgres_animals`
- **`partition_by_ingestion_date`** → Iceberg partition on the day of `_ingested_at`

In [ ]:
def build_bronze_resources():
    resources = []
    for name in DB_TABLES:
        r = sql_table(POSTGRES_URL, table=name,
                      incremental=dlt.sources.incremental("updated_at"))
        r.apply_hints(table_name=f"postgres_{name}")
        add_metadata(r, source_system="postgres", source_resource=name, ingestion_type="incremental")
        partition_by_ingestion_date(r)
        resources.append(r)
    return resources


bronze_pipeline = dlt.pipeline(
    pipeline_name="pile_incremental",
    destination=pile_destination("silo"),
    dataset_name=BRONZE,                 # dataset = namespace = medallion zone
    pipelines_dir=PIPELINES_DIR,
)

## Step 2 – First run (loads everything)

`run_to_pile(...)` = `pipeline.run(..., write_disposition="append", table_format="iceberg", loader_file_format="parquet")`, after telling dlt to use **our SQL catalog**.

In [ ]:
run_to_pile(bronze_pipeline, build_bronze_resources(), write_disposition="append")

### The tables are registered in the catalog

In [ ]:
print("Tables in bronze:", catalog.list_tables(BRONZE))
show_catalog_db("silo")

### The object keys

`bronze/<source>_<table>/data/ingestion_date=<YYYY-MM-DD>/<file>.parquet` = the data
`bronze/<source>_<table>/metadata/...` = Iceberg metadata (JSON = table description, `snap-*.avro` = snapshot, `*-m0.avro` = list of data files)

In [ ]:
list_pile("bronze/postgres_animals")

## Step 3 – Read with PyIceberg and verify

For every table: rows in the Iceberg table vs. rows in PostgreSQL, and the same ids.

In [ ]:
checks = []
for name in DB_TABLES:
    table = catalog.load_table(f"{BRONZE}.postgres_{name}")
    df = table.scan().to_pandas()
    db = pd.read_sql(f"SELECT id FROM {name}", engine)
    checks.append({
        "iceberg_table": f"{BRONZE}.postgres_{name}",
        "rows_iceberg": len(df),
        "rows_postgres": len(db),
        "same_ids": set(df.id) == set(db.id),
        "partition": str(table.spec()),
    })
pd.DataFrame(checks)

In [ ]:
animals = catalog.load_table(f"{BRONZE}.postgres_animals")
print(animals.schema())

In [ ]:
# PyIceberg can filter and select columns while reading
animals.scan(row_filter="species == 'dog'",
             selected_fields=("id", "name", "breed", "updated_at",
                              "_source_system", "_ingestion_type", "_ingested_at", "_load_id"),
             limit=5).to_pandas()

## Step 4 – Rerun without changes

Nothing is new → no rows, **no new snapshot**.

In [ ]:
run_to_pile(bronze_pipeline, build_bronze_resources(), write_disposition="append")
snapshots(catalog.load_table(f"{BRONZE}.postgres_animals"))

## Step 5 – Time travel

First remember the **current snapshot** (= the version "before").

In [ ]:
animals = catalog.load_table(f"{BRONZE}.postgres_animals")
snapshot_before = animals.current_snapshot().snapshot_id
print("Snapshot BEFORE:", snapshot_before)

**Now add some animals.**

- **Option A (best for the presentation):** open the app http://localhost:5173 → Animals → New, add 2–3 animals, then **skip** the next cell.
- **Option B:** run the next cell, it inserts 3 animals with SQL.

In [ ]:
# Option B – skip this cell if you added animals in the app
with engine.begin() as conn:
    new_ids = conn.execute(sa.text("""
        INSERT INTO animals (shelter_id, name, species, breed, sex, age_years, size, arrived_at)
        VALUES (1, 'Pile Test 1', 'dog', 'Beagle', 'M', 2, 'medium', CURRENT_DATE),
               (1, 'Pile Test 2', 'cat', 'Persian', 'F', 1, 'small', CURRENT_DATE),
               (2, 'Pile Test 3', 'rabbit', 'Holland Lop', 'F', 0.5, 'small', CURRENT_DATE)
        RETURNING id""")).scalars().all()
print("Inserted animal ids:", new_ids)

In [ ]:
run_to_pile(bronze_pipeline, build_bronze_resources(), write_disposition="append")

### The snapshots of the table

Every `append` created a new snapshot. `total_rows` grows, `schema_id` stays the same.

In [ ]:
animals = catalog.load_table(f"{BRONZE}.postgres_animals")
snapshots(animals)

### Read the table BEFORE and AFTER the change

`table.scan(snapshot_id=...)` reads the table **as it was** at that snapshot.

In [ ]:
snapshot_after = animals.current_snapshot().snapshot_id
before_df = animals.scan(snapshot_id=snapshot_before).to_pandas()
after_df = animals.scan(snapshot_id=snapshot_after).to_pandas()

print(f"BEFORE (snapshot {snapshot_before}): {len(before_df)} rows")
print(f"AFTER  (snapshot {snapshot_after}): {len(after_df)} rows")
print("\nRows that exist only AFTER the change:")
after_df[~after_df.id.isin(before_df.id)][["id", "name", "species", "updated_at", "_load_id"]]

In [ ]:
# Time travel by TIME instead of snapshot id: "what did the table look like at this moment?"
first_snapshot = animals.snapshots()[0]
as_of = animals.snapshot_as_of_timestamp(first_snapshot.timestamp_ms)
print("Snapshot valid at", pd.to_datetime(first_snapshot.timestamp_ms, unit="ms", utc=True), "→", as_of.snapshot_id)
print("Rows at that time:", len(animals.scan(snapshot_id=as_of.snapshot_id).to_arrow()))

In [ ]:
# The new snapshot only ADDED a small data file, the old one is untouched
data_files(animals)

## Bonus – incremental load into the additional pile (RustFS)

Same resources, a second pipeline (its own incremental state) and the **RustFS catalog**.

In [ ]:
ensure_pile_bucket("rustfs")
bronze_pipeline_rustfs = dlt.pipeline(
    pipeline_name="pile_incremental_rustfs",
    destination=pile_destination("rustfs"),
    dataset_name=BRONZE,
    pipelines_dir=PIPELINES_DIR,
)
run_to_pile(bronze_pipeline_rustfs, build_bronze_resources(), write_disposition="append", pile="rustfs")
show_catalog_db("rustfs")

In [ ]:
# Rerun → nothing new; and the same row counts as in Silo
run_to_pile(bronze_pipeline_rustfs, build_bronze_resources(), write_disposition="append", pile="rustfs")
catalog_rustfs = get_catalog("rustfs")
pd.DataFrame([{"table": f"postgres_{n}",
               "rows_silo": len(catalog.load_table(f"{BRONZE}.postgres_{n}").scan().to_arrow()),
               "rows_rustfs": len(catalog_rustfs.load_table(f"{BRONZE}.postgres_{n}").scan().to_arrow())}
              for n in DB_TABLES])